In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

project_root = Path.cwd().parents[2]

feature_dir = (
    project_root / "data" / "processed" / "engineered_features"
    / "group_3_other_skill_history" / "v3.3"
)

prediction_dir = (
    project_root / "data" / "processed" / "model_predictions"
)

prediction_dir.mkdir(parents=True, exist_ok=True)

print("Feature directory:", feature_dir)
print("Prediction directory:", prediction_dir)

Feature directory: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\engineered_features\group_3_other_skill_history\v3.3
Prediction directory: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions


In [2]:
v33_fold_datasets = []

for fold_number in range(1, 6):

    train_path = feature_dir / f"fold_{fold_number}_train.csv"
    val_path = feature_dir / f"fold_{fold_number}_validation.csv"

    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)

    v33_fold_datasets.append((train_df, val_df))

    print(
        f"Fold {fold_number}: "
        f"Train = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )

print("\nAll V3.3 datasets loaded successfully!")

Fold 1: Train = (177324, 5), Validation = (42659, 5)
Fold 2: Train = (176273, 5), Validation = (43710, 5)
Fold 3: Train = (170186, 5), Validation = (49797, 5)
Fold 4: Train = (178311, 5), Validation = (41672, 5)
Fold 5: Train = (177838, 5), Validation = (42145, 5)

All V3.3 datasets loaded successfully!


In [3]:
feature_columns = [
    "other_skill_smoothed_accuracy",
    "log_other_skill_previous_interactions"
]

target_column = "correct"

for fold_number, (train_df, val_df) in enumerate(
    v33_fold_datasets, start=1
):
    print(f"\n========== Fold {fold_number} ==========")

    print("Training missing values:", train_df.isna().sum().sum())
    print("Validation missing values:", val_df.isna().sum().sum())

    print("\nTraining feature ranges:")
    print(train_df[feature_columns].agg(["min", "max"]))

    print("\nValidation feature ranges:")
    print(val_df[feature_columns].agg(["min", "max"]))

    print("\nTraining target distribution:")
    print(train_df[target_column].value_counts(normalize=True))

    print("\nValidation target distribution:")
    print(val_df[target_column].value_counts(normalize=True))


========== Fold 1 ==========
Training missing values: 0
Validation missing values: 0

Training feature ranges:
     other_skill_smoothed_accuracy  log_other_skill_previous_interactions
min                       0.033701                               0.000000
max                       1.000000                               6.766192

Validation feature ranges:
     other_skill_smoothed_accuracy  log_other_skill_previous_interactions
min                       0.061581                               0.000000
max                       0.935119                               6.859615

Training target distribution:
correct
1    0.660136
0    0.339864
Name: proportion, dtype: float64

Validation target distribution:
correct
1    0.652336
0    0.347664
Name: proportion, dtype: float64

========== Fold 2 ==========
Training missing values: 0
Validation missing values: 0

Training feature ranges:
     other_skill_smoothed_accuracy  log_other_skill_previous_interactions
min                       0.

In [5]:
from sklearn.metrics import log_loss, brier_score_loss
feature_columns = [
    "other_skill_smoothed_accuracy",
    "log_other_skill_previous_interactions"
]

target_column = "correct"

v33_models = []
v33_fold_results = []
v33_predictions = []

for fold_number, (train_df, val_df) in enumerate(
    v33_fold_datasets, start=1
):
    print(f"\nTraining Fold {fold_number}...")

    # Separate features and target
    X_train = train_df[feature_columns]
    y_train = train_df[target_column]

    X_val = val_df[feature_columns]
    y_val = val_df[target_column]

    # Build Logistic Regression pipeline
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42
        ))
    ])

    # Train the model
    model.fit(X_train, y_train)

    # Predict probability of class 1 (correct)
    y_pred_proba = model.predict_proba(X_val)[:, 1]

    # Calculate fold-level primary metrics
    fold_log_loss = log_loss(y_val, y_pred_proba)
    fold_brier = brier_score_loss(y_val, y_pred_proba)

    # Store trained model
    v33_models.append(model)

    v33_fold_results.append({
        "fold": fold_number,
        "log_loss": fold_log_loss,
        "brier_score": fold_brier
    })

    # Store out-of-fold predictions
    fold_predictions = pd.DataFrame({
        "user_id": val_df["user_id"].values,
        "order_id": val_df["order_id"].values,
        "fold": fold_number,
        "y_true": y_val.values,
        "y_pred_proba": y_pred_proba,
        "model": "V3.3"
    })

    v33_predictions.append(fold_predictions)

    print(f"Log Loss: {fold_log_loss:.4f}")
    print(f"Brier Score: {fold_brier:.4f}")
    print(f"Predictions generated: {len(fold_predictions)}")


# Combine all five validation-fold predictions
v33_predictions_df = pd.concat(
    v33_predictions,
    ignore_index=True
)

# Save predictions to CSV
prediction_path = prediction_dir / "v3_3_predictions.csv"

v33_predictions_df.to_csv(
    prediction_path,
    index=False
)

print("\n========== V3.3 COMPLETED ==========")

print("\nAverage Log Loss:",
      np.mean([r["log_loss"] for r in v33_fold_results]))

print("Average Brier Score:",
      np.mean([r["brier_score"] for r in v33_fold_results]))

print("\nTotal OOF predictions:", len(v33_predictions_df))
print("Prediction file:", prediction_path)
print("\nAll five V3.3 models trained and predictions saved!")


Training Fold 1...
Log Loss: 0.6169
Brier Score: 0.2137
Predictions generated: 42659

Training Fold 2...
Log Loss: 0.6122
Brier Score: 0.2114
Predictions generated: 43710

Training Fold 3...
Log Loss: 0.6298
Brier Score: 0.2195
Predictions generated: 49797

Training Fold 4...
Log Loss: 0.6182
Brier Score: 0.2141
Predictions generated: 41672

Training Fold 5...
Log Loss: 0.6104
Brier Score: 0.2106
Predictions generated: 42145

========== V3.3 COMPLETED ==========

Average Log Loss: 0.6175056264011552
Average Brier Score: 0.21386362821619515

Total OOF predictions: 219983
Prediction file: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions\v3_3_predictions.csv

All five V3.3 models trained and predictions saved!
